# Preseason T–P


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio

GIMMS_PHENOLOGY_DIR = "../../data/satellite_data/images/PKU-GIMMS/phenology"
GIMMS_PHENOLOGY_PATTERN = "GIMMS_Phenology_SnowFilter_Forest1114_{year}.tif"

def load_gimms_snowfilter_sos_eos(df, years, phenology_dir=GIMMS_PHENOLOGY_DIR):
        coords = list(zip(df["longitude"].values, df["latitude"].values))
    n = len(coords)
    sample_year = next(
        y for y in years
        if os.path.exists(os.path.join(phenology_dir, GIMMS_PHENOLOGY_PATTERN.format(year=y)))
    )
    with rasterio.open(os.path.join(phenology_dir, GIMMS_PHENOLOGY_PATTERN.format(year=sample_year))) as src:
        transform = src.transform
        height, width = src.height, src.width

    rows = np.empty(n, dtype=np.int32)
    cols = np.empty(n, dtype=np.int32)
    for i, (lon, lat) in enumerate(coords):
        r, c = rasterio.transform.rowcol(transform, lon, lat)
        rows[i], cols[i] = r, c

    valid_rc = (rows >= 0) & (cols >= 0) & (rows < height) & (cols < width)

    for year in years:
        fp = os.path.join(phenology_dir, GIMMS_PHENOLOGY_PATTERN.format(year=year))
        if not os.path.exists(fp):
            print(f"  missing phenology file: {fp}", flush=True)
            df[f"sos_{year}"] = np.nan
            df[f"eos_{year}"] = np.nan
            continue
        with rasterio.open(fp) as src:
            sos_band = src.read(1)
            eos_band = src.read(2)
        sos = np.full(n, np.nan, dtype=np.float32)
        eos = np.full(n, np.nan, dtype=np.float32)
        sos[valid_rc] = sos_band[rows[valid_rc], cols[valid_rc]]
        eos[valid_rc] = eos_band[rows[valid_rc], cols[valid_rc]]
        sos[~np.isfinite(sos)] = np.nan
        eos[~np.isfinite(eos)] = np.nan
        df[f"sos_{year}"] = sos
        df[f"eos_{year}"] = eos
    return df

def _load_annual_climate_for_coords(lons, lats, years):
        key = set(zip(np.round(lons, 5), np.round(lats, 5)))
    years = set(int(y) for y in years)
    coords = pd.DataFrame({"longitude": lons, "latitude": lats})
    coords["_k"] = list(zip(np.round(coords["longitude"], 5), np.round(coords["latitude"], 5)))

    def load_chunked(paths, prefix):
        hits = []
        for fp in paths:
            cols = pd.read_csv(fp, nrows=0).columns
            ycols = [c for c in cols if c.startswith(prefix) and int(c.split("_")[-1]) in years]
            if not ycols:
                continue
            usecols = ["longitude", "latitude"] + ycols
            for chunk in pd.read_csv(fp, usecols=usecols, chunksize=200_000):
                mask = [
                    (round(lo, 5), round(la, 5)) in key
                    for lo, la in zip(chunk["longitude"], chunk["latitude"])
                ]
                sub = chunk.loc[mask]
                if len(sub):
                    hits.append(sub)
        if not hits:
            return coords[["longitude", "latitude"]].copy()
        # Period files have disjoint year columns; combine per-pixel (do NOT
        # drop_duplicates — that kept only 1982–1999 and left 2000+ as NaN).
        df = pd.concat(hits, ignore_index=True)
        df["_k"] = list(zip(np.round(df["longitude"], 5), np.round(df["latitude"], 5)))
        ycols = [c for c in df.columns if c.startswith(prefix)]
        df = df.groupby("_k", as_index=False)[ycols].first()
        return coords[["_k"]].merge(df, on="_k", how="left")

    t_paths = [
        "../../data/climate_data/tables/climate_data/temp/temp-1982-1999.csv",
        "../../data/climate_data/tables/climate_data/temp/temp-2000-2024.csv",
    ]
    p_paths = [
        "../../data/climate_data/tables/climate_data/prcp/prcp-1982-1999.csv",
        "../../data/climate_data/tables/climate_data/prcp/prcp-2000-2024.csv",
    ]
    print("  loading annual T from climate tables...", flush=True)
    tdf = load_chunked(t_paths, "annual_t_")
    print("  loading annual P from climate tables...", flush=True)
    pdf = load_chunked(p_paths, "annual_p_")
    out = coords[["longitude", "latitude"]].copy()
    for c in tdf.columns:
        if c.startswith("annual_t_"):
            out[c] = tdf[c].values
    for c in pdf.columns:
        if c.startswith("annual_p_"):
            out[c] = pdf[c].values
    return out

def read_satellite_data(veg_type, satellite):
    veg_class = pd.read_csv("../../data/veg_class_data/tables/veg_class.csv")
    clim_fp = f"../../data/satellite_data/tables/phenology_climate/{satellite}.csv"
    if satellite == "gimms" and not os.path.exists(clim_fp):
        print(f"  {clim_fp} missing — rebuilding annual T/P from climate tables", flush=True)
        forest = veg_class[veg_class["veg_class"].isin([11, 12, 13, 14])].copy()
        if veg_type in (11, 12, 13, 14):
            forest = forest[forest["veg_class"] == veg_type].copy()
        years = list(range(1982, 2023))
        df_satellite = _load_annual_climate_for_coords(
            forest["longitude"].values, forest["latitude"].values, years
        )
        df = forest.merge(df_satellite, on=["longitude", "latitude"], how="inner")
    else:
        df_satellite = pd.read_csv(clim_fp)
        df = pd.merge(df_satellite, veg_class, on=["latitude", "longitude"], how="inner")
    if veg_type in (11, 12, 13, 14):
        df = df[df["veg_class"].isin([veg_type])]
    else:
        df = df[df["veg_class"].isin([11, 12, 13, 14])]

    eos_cols = [col for col in df.columns if "eos" in col]
    t_cols = [col for col in df.columns if "annual_t" in col]
    p_cols = [col for col in df.columns if "annual_p" in col]
    sos_cols = [col for col in df.columns if "sos" in col]

    if satellite == "gimms":
        years = [str(y) for y in range(1982, 2023)]
        df = df.drop(columns=[c for c in eos_cols + sos_cols], errors="ignore")
        df = load_gimms_snowfilter_sos_eos(df, [int(y) for y in years])
        eos_cols = [col for col in df.columns if col.startswith("eos_")]
        sos_cols = [col for col in df.columns if col.startswith("sos_")]
    elif satellite == "avhrr":
        years = [str(y) for y in range(1982, 2017)]
    elif satellite == "modis":
        years = [str(y) for y in range(2001, 2024)]
        mask_sos = (df[sos_cols] < 0).any(axis=1)
        mask_eos = (df[eos_cols] > 365).any(axis=1)
        df = df[~(mask_sos | mask_eos)].copy()
    else:
        years = [str(y) for y in range(2013, 2023)]
        mask_sos = (df[sos_cols] < 0).any(axis=1)
        mask_eos = (df[eos_cols] > 365).any(axis=1)
        df = df[~(mask_sos | mask_eos)].copy()

    cols = years
    df = df[[col for col in eos_cols + t_cols + p_cols + sos_cols if any(y in col for y in cols)] + ["latitude", "longitude", "veg_class"]].copy()
    t_cols_df = [col for col in df.columns if "annual_t" in col]
    df[t_cols_df] = df[t_cols_df] - 273.5  # Convert temperature
    df.columns = df.columns.str.replace(r"\D*(\d{4})$", lambda m: f"{m.group(0)[0:-4]}{m.group(1)}", regex=True)
    df["annual_t"] = df[[col for col in df.columns if "annual_t" in col]].mean(axis=1)
    df["annual_p"] = df[[col for col in df.columns if "annual_p" in col]].mean(axis=1)
    eos_year_cols = [col for col in df.columns if col.startswith("eos_")]
    sos_year_cols = [col for col in df.columns if col.startswith("sos_")]
    df["eos"] = df[eos_year_cols].mean(axis=1)
    df["sos"] = df[sos_year_cols].mean(axis=1)
    if satellite == "gimms":
        df = df[df["eos"].notna()].copy()
    return df


In [ ]:
import glob
import time
from collections import defaultdict

from rasterio.transform import rowcol
from rasterio.windows import Window
from scipy.stats import pearsonr

CLIMATE_ROOT = "../../data/climate_data/images/daily_climate-1982-2024"
YEARS_PRE = list(range(1982, 2023))
MONTH_DAYS = {m: 30 * m for m in range(1, 10)}  # max 9 months
MAX_PRESEASON_DAYS = 270
CLIMATE_VARS_TP = {"t": "daily_t", "p": "daily_p"}
CLIMATE_VARS_R = {"r": "daily_r"}
PRESEASON_CACHE_DIR = "../../results/figure2/gimms/preseason_cache"
os.makedirs(PRESEASON_CACHE_DIR, exist_ok=True)
FP_PRESEASON_EXTRACT = os.path.join(PRESEASON_CACHE_DIR, "forest_preseason_mean_eos_9m.npz")
FP_PRESEASON_META = os.path.join(PRESEASON_CACHE_DIR, "forest_preseason_meta.csv")
FP_PRESEASON_SELECTED = os.path.join(PRESEASON_CACHE_DIR, "forest_preseason_selected_tp.npz")
FP_RAD_EXTRACT = os.path.join(PRESEASON_CACHE_DIR, "all_pixels_rad_mean_eos_9m.npz")
FP_SOS_EXTRACT = os.path.join(PRESEASON_CACHE_DIR, "all_pixels_sos.npz")
FORCE_REEXTRACT_PRESEASON = False  # True only to rebuild daily extracts
FP_LENGTHS = os.path.join(PRESEASON_CACHE_DIR, "all_pixels_fig2_preseason_lengths.csv")

def build_tile_index(year=2000):
    tiles = []
    folder = os.path.join(CLIMATE_ROOT, "daily_t", f"daily_t_{year}")
    for fp in sorted(glob.glob(os.path.join(folder, "*.tif"))):
        base = os.path.basename(fp)
        tile_key = base.split(str(year), 1)[1]
        with rasterio.open(fp) as src:
            tiles.append({
                "bounds": src.bounds,
                "transform": src.transform,
                "height": src.height,
                "width": src.width,
                "tile_key": tile_key,
            })
    return tiles

def assign_pixels_to_tiles(lons, lats, tiles):
    n = len(lons)
    out = [None] * n
    lons = np.asarray(lons, dtype=float)
    lats = np.asarray(lats, dtype=float)
    for i, t in enumerate(tiles):
        b = t["bounds"]
        mask = (lons >= b.left) & (lons <= b.right) & (lats >= b.bottom) & (lats <= b.top)
        indices = np.where(mask)[0]
        for idx in indices:
            if out[idx] is None:
                r, c = rowcol(t["transform"], lons[idx], lats[idx])
                if 0 <= r < t["height"] and 0 <= c < t["width"]:
                    out[idx] = (i, int(r), int(c))
    return out

def find_tile_file(var_dir_name, year, tile_key):
    fp = os.path.join(CLIMATE_ROOT, var_dir_name, f"{var_dir_name}_{year}", f"{var_dir_name}_{year}{tile_key}")
    return fp if os.path.exists(fp) else None

def simple_corr(x, y):
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 8 or np.nanstd(x[mask]) == 0 or np.nanstd(y[mask]) == 0:
        return np.nan
    r, _ = pearsonr(x[mask], y[mask])
    return float(r)

def extract_preseason_mean_eos(df, eos_mat, climate_vars, agg_by_var=None, years=YEARS_PRE):
    if agg_by_var is None:
        agg_by_var = {v: ("sum" if v == "p" else "mean") for v in climate_vars}

    lons = df["longitude"].to_numpy()
    lats = df["latitude"].to_numpy()
    n, nyears = len(df), len(years)
    mean_eos = np.nanmean(eos_mat, axis=1)
    tile_meta = build_tile_index(2000)
    pixel_tile_rc = assign_pixels_to_tiles(lons, lats, tile_meta)
    print("  on climate tiles:", sum(x is not None for x in pixel_tile_rc), "/", n, flush=True)

    preseason = {v: {m: np.full((n, nyears), np.nan, dtype=np.float32) for m in MONTH_DAYS} for v in climate_vars}
    t0 = time.time()
    for yi, year in enumerate(years):
        print(f"  {year} ({yi+1}/{nyears})  elapsed={time.time()-t0:.0f}s", flush=True)
        by_tile = defaultdict(list)
        for i, loc in enumerate(pixel_tile_rc):
            if loc is None or not np.isfinite(mean_eos[i]):
                continue
            by_tile[loc[0]].append(i)

        for var, var_dir in climate_vars.items():
            for tile_idx, idxs in by_tile.items():
                fp = find_tile_file(var_dir, year, tile_meta[tile_idx]["tile_key"])
                if fp is None:
                    continue
                idxs = np.asarray(idxs, dtype=np.int64)
                ends = np.floor(mean_eos[idxs]).astype(int)
                starts = np.maximum(1, ends - MAX_PRESEASON_DAYS)
                b0, b1 = int(starts.min()), int(ends.max())
                if b1 <= b0:
                    continue
                with rasterio.open(fp) as src:
                    b1_read = min(int(b1), src.count + 1)
                    if b1_read <= b0:
                        continue
                    stack = src.read(list(range(b0, b1_read)), out_dtype="float32")
                rows = np.array([pixel_tile_rc[i][1] for i in idxs])
                cols = np.array([pixel_tile_rc[i][2] for i in idxs])
                vals = stack[:, rows, cols].astype(np.float64, copy=False)
                if var == "t":
                    vals = vals - 273.15
                valid = np.isfinite(vals)
                vals0 = np.where(valid, vals, 0.0)
                csum = np.cumsum(vals0, axis=0)
                ccnt = np.cumsum(valid.astype(np.float64), axis=0)
                z = np.zeros((1, vals.shape[1]), dtype=np.float64)
                csum = np.vstack([z, csum])
                ccnt = np.vstack([z, ccnt])
                for m, ndays in MONTH_DAYS.items():
                    s = np.maximum(1, ends - ndays)
                    i0 = s - b0
                    i1 = ends - b0
                    ok = (i0 >= 0) & (i1 > i0) & (i1 <= vals.shape[0])
                    if not np.any(ok):
                        continue
                    jj = np.where(ok)[0]
                    sm = csum[i1[jj], jj] - csum[i0[jj], jj]
                    ct = ccnt[i1[jj], jj] - ccnt[i0[jj], jj]
                    good = ct > 0
                    jj = jj[good]
                    sm = sm[good]
                    ct = ct[good]
                    pix = idxs[jj]
                    if agg_by_var.get(var, "mean") == "sum":
                        preseason[var][m][pix, yi] = sm.astype(np.float32)
                    else:
                        preseason[var][m][pix, yi] = (sm / ct).astype(np.float32)
    print(f"  extract done in {(time.time()-t0)/60:.1f} min", flush=True)
    return preseason

def extract_preseason_mean_eos_tp(df, eos_mat, years=YEARS_PRE):
        return extract_preseason_mean_eos(df, eos_mat, CLIMATE_VARS_TP, years=years)

def select_preseason_series(eos_mat, preseason, var):
        n, nyears = eos_mat.shape
    out = np.full((n, nyears), np.nan, dtype=np.float32)
    Larr = np.full(n, np.nan)
    for i in range(n):
        eos = eos_mat[i]
        best_m, best_abs = np.nan, -np.inf
        for m in MONTH_DAYS:
            rr = simple_corr(preseason[var][m][i], eos)
            if np.isfinite(rr) and abs(rr) > best_abs:
                best_abs, best_m = abs(rr), m
        if np.isfinite(best_m):
            Larr[i] = best_m
            out[i] = preseason[var][int(best_m)][i]
    return out, Larr

def select_preseason_tp_series(eos_mat, preseason):
        t_sel, L_t = select_preseason_series(eos_mat, preseason, "t")
    p_sel, L_p = select_preseason_series(eos_mat, preseason, "p")
    return t_sel, p_sel, L_t, L_p

def attach_preseason_to_df(df, t_sel, p_sel, L_t, L_p, years=YEARS_PRE):
    df = df.copy()
    for yi, year in enumerate(years):
        df[f"preseason_t_{year}"] = t_sel[:, yi]
        df[f"preseason_p_{year}"] = p_sel[:, yi]
    df["preseason_t_months"] = L_t
    df["preseason_p_months"] = L_p
    return df



In [ ]:
import re

def cal_anomalies(df):
        df = df.copy()
    anom_parts = []

    eos_cols = sorted(
        [c for c in df.columns if re.fullmatch(r"eos_\d{4}", c)],
        key=lambda c: c.split("_")[1],
    )
    t_cols = sorted(
        [c for c in df.columns if re.fullmatch(r"preseason_t_\d{4}", c)],
        key=lambda c: c.split("_")[2],
    )
    p_cols = sorted(
        [c for c in df.columns if re.fullmatch(r"preseason_p_\d{4}", c)],
        key=lambda c: c.split("_")[2],
    )

    df["eos"] = df[eos_cols].mean(axis=1, skipna=True)
    df["preseason_t"] = df[t_cols].mean(axis=1, skipna=True)
    df["preseason_p"] = df[p_cols].mean(axis=1, skipna=True)

    eos_anom = df[eos_cols].sub(df["eos"], axis=0)
    eos_anom.columns = [f"eos_anom_{c.split('_')[1]}" for c in eos_cols]
    anom_parts.append(eos_anom)

    t_anom = df[t_cols].sub(df["preseason_t"], axis=0)
    t_anom.columns = [f"preseason_t_anom_{c.split('_')[2]}" for c in t_cols]
    anom_parts.append(t_anom)

    # preseason_p stored in meters (window sum); anomaly in mm
    p_anom = df[p_cols].sub(df["preseason_p"], axis=0) * 1000
    p_anom.columns = [f"preseason_p_anom_{c.split('_')[2]}" for c in p_cols]
    anom_parts.append(p_anom)

    anom = pd.concat(anom_parts, axis=1)
    df = pd.concat([df, anom], axis=1)
    return df


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import binned_statistic_2d
from matplotlib.colors import ListedColormap
from matplotlib.ticker import MultipleLocator

def plot_eos_anom_all_years(df,
                            t_bins=np.arange(-3.8, 3.9, 0.1),
                            # include +400 as the upper edge (arange stop is exclusive)
                            p_bins=np.arange(-400, 420, 20),
                            trim=True, lower=5, upper=95, min_count=10,
                            color_by="eos_mean",
                            pct_by="raw"):
    """
    2D preseason T–P anomaly grid.
    color_by: "eos_mean" | "count"
    pct_by: "raw" | "grid"
    """
    eos_cols = [c for c in df.columns if c.startswith("eos_anom_")]
    t_cols   = [c for c in df.columns if c.startswith("preseason_t_anom_")]
    p_cols   = [c for c in df.columns if c.startswith("preseason_p_anom_")]

    eos_anom = df[eos_cols].to_numpy().ravel()
    t_anom   = df[t_cols].to_numpy().ravel()
    p_anom   = df[p_cols].to_numpy().ravel()

    mask = ~np.isnan(eos_anom) & ~np.isnan(t_anom) & ~np.isnan(p_anom)
    eos_anom, t_anom, p_anom = eos_anom[mask], t_anom[mask], p_anom[mask]

    if trim and len(eos_anom):
        e_low, e_high = np.percentile(eos_anom, [lower, upper])
        mask = (eos_anom >= e_low) & (eos_anom <= e_high)
        eos_anom, t_anom, p_anom = eos_anom[mask], t_anom[mask], p_anom[mask]

    stat, x_edge, y_edge, _ = binned_statistic_2d(
        t_anom, p_anom, eos_anom, statistic="mean", bins=[t_bins, p_bins]
    )
    count, _, _, _ = binned_statistic_2d(
        t_anom, p_anom, eos_anom, statistic="count", bins=[t_bins, p_bins]
    )

    keep_bin = count >= min_count
    stat = stat.copy()
    stat[~keep_bin] = np.nan
    count_plot = count.astype(float).copy()
    count_plot[~keep_bin] = np.nan

    t_idx = np.digitize(t_anom, x_edge) - 1
    p_idx = np.digitize(p_anom, y_edge) - 1
    in_grid = (
        (t_idx >= 0) & (t_idx < stat.shape[0]) &
        (p_idx >= 0) & (p_idx < stat.shape[1])
    )
    in_filt = np.zeros(len(eos_anom), dtype=bool)
    in_filt[in_grid] = keep_bin[t_idx[in_grid], p_idx[in_grid]]
    eos_f = eos_anom[in_filt]
    t_f = t_anom[in_filt]
    p_f = p_anom[in_filt]

    fig, ax = plt.subplots(figsize=(6, 6))
    from matplotlib import colors

    if color_by == "count":
        plot_stat = count_plot
        finite = plot_stat[np.isfinite(plot_stat)]
        vmax = float(np.percentile(finite, 95)) if finite.size else 1.0
        vmax = max(vmax, 1.0)
        cmap = plt.cm.YlOrRd
        norm = colors.Normalize(vmin=0, vmax=vmax, clip=True)
        pcm = ax.pcolormesh(x_edge, y_edge, plot_stat.T, cmap=cmap, norm=norm, shading="auto", zorder=0)
        cbar = fig.colorbar(pcm, ax=ax, orientation="horizontal", pad=0.2, shrink=0.8)
        cbar.set_label("Sample count per grid cell", fontsize=14)
        cbar.ax.tick_params(labelsize=12)
    else:
        cmap = ListedColormap(
            ["#0b3c68", "#165188", "#2066a8", "#4d91c4", "#8ec1da",
             "#fbebe1", "#f6d6c2", "#d47264", "#c14d48", "#ae282c"]
        )
        norm = colors.Normalize(vmin=-5, vmax=5, clip=True)
        pcm = ax.pcolormesh(x_edge, y_edge, stat.T, cmap=cmap, norm=norm, shading="auto", zorder=0)
        cbar = fig.colorbar(pcm, ax=ax, orientation="horizontal", pad=0.2, shrink=0.8)
        cbar.set_label("EOS anomaly (days)", fontsize=14)
        cbar.set_ticks([-4, -3, -2, -1, 0, 1, 2, 3, 4])
        cbar.ax.tick_params(labelsize=12)

    ax.set_xlabel("Preseason T anomaly (°C)", fontsize=12)
    ax.set_ylabel("Preseason P anomaly (mm/year)", fontsize=12)
    ax.set_xlim(-3.8, 3.8)
    ax.set_ylim(-400, 400)
    ax.yaxis.set_major_locator(MultipleLocator(200))
    ax.axvline(0, color="black", linestyle="--", linewidth=1)
    ax.axhline(0, color="black", linestyle="--", linewidth=1)
    ax.tick_params(labelsize=12)

    text_positions = {
        "T>0, P>0": (0.93, 0.90),
        "T>0, P<0": (0.93, 0.10),
        "T<0, P>0": (0.07, 0.90),
        "T<0, P<0": (0.07, 0.10),
    }
    bar_colors = {"Positive": "#d47264", "Negative": "#2066a8"}
    from matplotlib.patches import Rectangle

    def _draw_pct(pos_pct, neg_pct, label):
        x_pos, y_pos = text_positions[label]
        rect = Rectangle(
            (x_pos - 0.11, y_pos - 0.07), 0.15, 0.15,
            transform=ax.transAxes, color="white", zorder=1,
        )
        ax.add_patch(rect)
        ax.text(
            x_pos, y_pos + 0.01, f"{pos_pct:.1f}%",
            transform=ax.transAxes, fontsize=13,
            ha="center", va="bottom", color=bar_colors["Positive"], fontweight="bold",
        )
        ax.text(
            x_pos, y_pos - 0.03, f"{neg_pct:.1f}%",
            transform=ax.transAxes, fontsize=13,
            ha="center", va="top", color=bar_colors["Negative"], fontweight="bold",
        )

    if pct_by == "grid":
        t_centers = (x_edge[:-1] + x_edge[1:]) / 2
        p_centers = (y_edge[:-1] + y_edge[1:]) / 2
        T_grid, P_grid = np.meshgrid(t_centers, p_centers, indexing="ij")
        quadrants = {
            "T>0, P>0": (T_grid > 0) & (P_grid > 0),
            "T>0, P<0": (T_grid > 0) & (P_grid < 0),
            "T<0, P>0": (T_grid < 0) & (P_grid > 0),
            "T<0, P<0": (T_grid < 0) & (P_grid < 0),
        }
        for label, qmask in quadrants.items():
            vals = stat[qmask]
            vals = vals[np.isfinite(vals)]
            if len(vals) == 0:
                pos_pct = neg_pct = 0.0
            else:
                pos_pct = 100.0 * np.mean(vals > 0)
                neg_pct = 100.0 * np.mean(vals < 0)
            _draw_pct(pos_pct, neg_pct, label)
    else:
        quadrants = {
            "T>0, P>0": (t_f > 0) & (p_f > 0),
            "T>0, P<0": (t_f > 0) & (p_f < 0),
            "T<0, P>0": (t_f < 0) & (p_f > 0),
            "T<0, P<0": (t_f < 0) & (p_f < 0),
        }
        for label, qmask in quadrants.items():
            vals = eos_f[qmask]
            if len(vals) == 0:
                pos_pct = neg_pct = 0.0
            else:
                pos_pct = 100.0 * np.mean(vals > 0)
                neg_pct = 100.0 * np.mean(vals < 0)
            _draw_pct(pos_pct, neg_pct, label)

    return fig



In [ ]:
satellite = 'gimms'

veg_type = 0
df = read_satellite_data(veg_type, satellite)

In [ ]:
eos_cols = sorted([c for c in df.columns if c.startswith("eos_") and c[4:].isdigit()], key=lambda c: int(c.split("_")[1]))
years_eos = [int(c.split("_")[1]) for c in eos_cols]
assert years_eos == YEARS_PRE, (years_eos[:3], years_eos[-1], len(years_eos))
eos_mat = df[eos_cols].to_numpy(dtype=float)

def _coords_match(lat, lon, df_ref, atol=1e-6):
    return (
        len(lat) == len(df_ref)
        and np.allclose(lat, df_ref["latitude"].values, atol=atol, rtol=0)
        and np.allclose(lon, df_ref["longitude"].values, atol=atol, rtol=0)
    )

def _years_match(years_arr):
    return [int(y) for y in np.asarray(years_arr).tolist()] == YEARS_PRE

t_sel = p_sel = L_t = L_p = None

if (not FORCE_REEXTRACT_PRESEASON) and os.path.exists(FP_PRESEASON_SELECTED):
    zs = np.load(FP_PRESEASON_SELECTED)
    if (
        _years_match(zs["years"])
        and _coords_match(zs["latitude"], zs["longitude"], df)
        and zs["t_sel"].shape == (len(df), len(YEARS_PRE))
        and zs["p_sel"].shape == (len(df), len(YEARS_PRE))
    ):
        print("Using cached selected preseason T/P:", FP_PRESEASON_SELECTED)
        t_sel = zs["t_sel"]
        p_sel = zs["p_sel"]
        L_t = zs["L_t"]
        L_p = zs["L_p"]
    else:
        print("Selected-series cache mismatch — will try 1–9m extract")

preseason = None
if t_sel is None and (not FORCE_REEXTRACT_PRESEASON) and os.path.exists(FP_PRESEASON_EXTRACT):
    z = np.load(FP_PRESEASON_EXTRACT)
    lat = z["latitude"] if "latitude" in z.files else None
    lon = z["longitude"] if "longitude" in z.files else None
    if lat is None or lon is None:
        if os.path.exists(FP_PRESEASON_META):
            meta = pd.read_csv(FP_PRESEASON_META)
            lat, lon = meta["latitude"].values, meta["longitude"].values
    ok = (
        lat is not None
        and _years_match(z["years"])
        and _coords_match(lat, lon, df)
        and all(f"t_{m}m" in z.files for m in MONTH_DAYS)
        and all(f"p_{m}m" in z.files for m in MONTH_DAYS)
        and z["t_1m"].shape[0] == len(df)
    )
    if ok:
        print("Using cached preseason extract:", FP_PRESEASON_EXTRACT)
        preseason = {v: {m: z[f"{v}_{m}m"] for m in MONTH_DAYS} for v in CLIMATE_VARS_TP}
    else:
        print("Extract cache mismatch — will re-extract from daily climate")
        print("  n_cache=", None if lat is None else len(lat), "n_df=", len(df))

if t_sel is None and preseason is None:
    print("Extracting mean-EOS preseason T/P for", len(df), "pixels ×", len(YEARS_PRE), "years...")
    preseason = extract_preseason_mean_eos_tp(df, eos_mat)
    save_kw = {
        "years": np.array(YEARS_PRE),
        "eos": eos_mat.astype(np.float32),
        "latitude": df["latitude"].to_numpy(np.float64),
        "longitude": df["longitude"].to_numpy(np.float64),
    }
    for v in CLIMATE_VARS_TP:
        for m in MONTH_DAYS:
            save_kw[f"{v}_{m}m"] = preseason[v][m]
    np.savez_compressed(FP_PRESEASON_EXTRACT, **save_kw)
    df[["latitude", "longitude", "veg_class", "annual_t", "annual_p"]].to_csv(FP_PRESEASON_META, index=False)
    print("Saved", FP_PRESEASON_EXTRACT)

if t_sel is None:
    t_sel, p_sel, L_t, L_p = select_preseason_tp_series(eos_mat, preseason)
    np.savez_compressed(
        FP_PRESEASON_SELECTED,
        years=np.array(YEARS_PRE),
        latitude=df["latitude"].to_numpy(np.float64),
        longitude=df["longitude"].to_numpy(np.float64),
        t_sel=np.asarray(t_sel, dtype=np.float32),
        p_sel=np.asarray(p_sel, dtype=np.float32),
        L_t=np.asarray(L_t),
        L_p=np.asarray(L_p),
    )
    print("Saved", FP_PRESEASON_SELECTED)

df = attach_preseason_to_df(df, t_sel, p_sel, L_t, L_p)
print("Selected length means: T={:.2f} mo, P={:.2f} mo".format(np.nanmean(L_t), np.nanmean(L_p)))
print("Finite frac preseason_t:", np.isfinite(t_sel).mean(), "preseason_p:", np.isfinite(p_sel).mean())
print(df[["preseason_t_months", "preseason_p_months"]].describe().round(2))


In [ ]:

sos_cols = sorted(
    [c for c in df.columns if c.startswith("sos_") and c[4:].isdigit()],
    key=lambda c: int(c.split("_")[1]),
)
assert [int(c.split("_")[1]) for c in sos_cols] == YEARS_PRE
sos_mat = df[sos_cols].to_numpy(dtype=float)

need_r = True
if (not FORCE_REEXTRACT_PRESEASON) and os.path.exists(FP_RAD_EXTRACT):
    zr = np.load(FP_RAD_EXTRACT)
    lat_ok = ("latitude" not in zr.files) or _coords_match(zr["latitude"], zr["longitude"], df)
    if (
        _years_match(zr["years"])
        and zr["r_1m"].shape[0] == len(df)
        and lat_ok
        and all(f"r_{m}m" in zr.files for m in MONTH_DAYS)
    ):
        print("Using cached preseason R:", FP_RAD_EXTRACT)
        preseason_r = {"r": {m: zr[f"r_{m}m"] for m in MONTH_DAYS}}
        need_r = False
    else:
        print("R cache mismatch — re-extracting")
        print("  n_cache=", zr["r_1m"].shape[0], "n_df=", len(df), "lat_ok=", lat_ok)

if need_r:
    print("Extracting mean-EOS preseason R for", len(df), "pixels ×", len(YEARS_PRE), "years...")
    preseason_r = extract_preseason_mean_eos(df, eos_mat, CLIMATE_VARS_R, agg_by_var={"r": "mean"})
    save_r = {
        "years": np.array(YEARS_PRE),
        "latitude": df["latitude"].to_numpy(np.float64),
        "longitude": df["longitude"].to_numpy(np.float64),
    }
    for m in MONTH_DAYS:
        save_r[f"r_{m}m"] = preseason_r["r"][m]
    np.savez_compressed(FP_RAD_EXTRACT, **save_r)
    print("Saved", FP_RAD_EXTRACT)

r_sel, L_r = select_preseason_series(eos_mat, preseason_r, "r")
df["preseason_r_months"] = L_r
for yi, year in enumerate(YEARS_PRE):
    df[f"preseason_r_{year}"] = r_sel[:, yi]
print(
    "Selected length means: T={:.2f}, P={:.2f}, R={:.2f} mo".format(
        np.nanmean(df["preseason_t_months"]),
        np.nanmean(df["preseason_p_months"]),
        np.nanmean(L_r),
    )
)
print("Finite frac preseason_r:", np.isfinite(r_sel).mean())

np.savez_compressed(
    FP_SOS_EXTRACT,
    years=np.array(YEARS_PRE),
    sos=sos_mat.astype(np.float32),
    latitude=df["latitude"].to_numpy(np.float64),
    longitude=df["longitude"].to_numpy(np.float64),
)
print("Saved", FP_SOS_EXTRACT)

leng = df[["latitude", "longitude", "preseason_t_months", "preseason_p_months", "preseason_r_months"]].copy()
leng.to_csv(FP_LENGTHS, index=False)
print("Saved", FP_LENGTHS)
print(leng[["preseason_t_months", "preseason_p_months", "preseason_r_months"]].describe().round(2))


In [ ]:
import os
out_dir = f"../../results/figure2/{satellite}/anomaly_preseason_gridpct"
os.makedirs(out_dir, exist_ok=True)

# min_count=10: drop T–P bins with fewer than 10 samples
kw = dict(pct_by="grid", min_count=10)

df_all = cal_anomalies(df)
fig = plot_eos_anom_all_years(df_all, **kw)
fig.suptitle("All forests — EOS anomaly (grid %, preseason T–P)", y=1.02)
fig.savefig(f"{out_dir}/all.png", dpi=300, bbox_inches="tight")
plt.show()

df_hot_dry = df[(df["annual_t"] > 7.25) & (df["annual_p"] < 1.0)].copy()
df_hot_dry = cal_anomalies(df_hot_dry)
fig = plot_eos_anom_all_years(df_hot_dry, **kw)
fig.suptitle("Hot-dry — EOS anomaly (grid %, preseason T–P)", y=1.02)
fig.savefig(f"{out_dir}/hot-dry.png", dpi=300, bbox_inches="tight")
plt.show()

df_cold_dry = df[(df["annual_t"] < 7.25) & (df["annual_p"] < 1.0)].copy()
df_cold_dry = cal_anomalies(df_cold_dry)
fig = plot_eos_anom_all_years(df_cold_dry, **kw)
fig.suptitle("Cold-dry — EOS anomaly (grid %, preseason T–P)", y=1.02)
fig.savefig(f"{out_dir}/cold-dry.png", dpi=300, bbox_inches="tight")
plt.show()

df_wet = df[(df["annual_p"] >= 1.0)].copy()
df_wet = cal_anomalies(df_wet)
fig = plot_eos_anom_all_years(df_wet, **kw)
fig.suptitle("Wet — EOS anomaly (grid %, preseason T–P)", y=1.02)
fig.savefig(f"{out_dir}/wet.png", dpi=300, bbox_inches="tight")
plt.show()
